<a href="https://colab.research.google.com/github/1kaiser/opyf_colab/blob/main/jax_3d_reconstruction_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JAX 3D Reconstruction Pipeline (Zonal) 🚀\nThis notebook provides a high-performance JAX ecosystem for metric 3D reconstruction.\n\nAll code and datasets are now integrated directly into the **opyf_colab** repository.

## 1. Environment Setup

In [ ]:
# @title 1.1 Setup & Dependencies
import sys
import os

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()

if IN_COLAB:
    print("Running in Google Colab")
    os.system("pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html")
    os.system("pip install -q flax open3d tqdm opencv-python")
    
    if not os.path.exists("/content/opyf_colab"):
        os.system("git clone https://github.com/1kaiser/opyf_colab.git /content/opyf_colab")
    
    sys.path.append("/content/opyf_colab")
    sys.path.append("/content/opyf_colab/models/jax")
    os.chdir("/content/opyf_colab")
else:
    print("Running Locally")
    project_root = os.getcwd() 
    if project_root not in sys.path:
        sys.path.append(project_root)
    models_path = os.path.join(project_root, "models/jax")
    if models_path not in sys.path:
        sys.path.append(models_path)

import jax
import flax
import open3d as o3d
from tqdm import tqdm
import cv2
print(f"JAX Devices: {jax.devices()}")


In [ ]:
# @title 1.2 Weights Management
import os

def download_weights():
    os.makedirs("weights", exist_ok=True)
    BASE_URL = "https://github.com/1kaiser/d_jax/releases/download/v1.0.0"

    print("Downloading/Verifying weights...")
    
    models = {
        "depth_pro.msgpack": None,
        "superpoint.msgpack": None,
        "superpoint_lightglue.msgpack": None
    }

    for target, parts in models.items():
        target_path = os.path.join("weights", target)
        if os.path.exists(target_path):
            print(f"✅ {target} already exists.")
            continue
            
        print(f"Downloading {target}...")
        url = f"{BASE_URL}/{target}"
        if IN_COLAB:
            os.system(f"wget -q --show-progress {url} -O {target_path}")
        else:
            import subprocess
            subprocess.run(["wget", "-q", "--show-progress", url, "-O", target_path])

    print("\n✅ Required weights ready.")

if IN_COLAB:
    download_weights()
else:
    required = ["depth_pro.msgpack", "superpoint.msgpack", "superpoint_lightglue.msgpack"]
    missing = [r for r in required if not os.path.exists(os.path.join("weights", r))]
    if missing:
        print(f"Missing weights: {missing}. Downloading...")
        download_weights()
    else:
        print("✅ Local weights detected.")


## 2. Standalone Model Testing

In [ ]:
if IN_COLAB:\n    test_img1 = "./data/pinecone_subset/frame_001.jpg"\nelse:\n    test_img1 = "./data/pinecone_subset/frame_001.jpg"\nos.makedirs("output", exist_ok=True)\nos.system(f"python -m inference.infer_depth_pro --image {test_img1} --weights weights/depth_pro.msgpack --output output/depth_result.jpg")

## 3. Zonal 3D Reconstruction

In [ ]:
from pipelines.pipeline_jax import ReconstructionPipeline\npipeline = ReconstructionPipeline("weights/depth_pro.msgpack", "weights/superpoint.msgpack", "weights/superpoint_lightglue.msgpack")\nT_globals = pipeline.run("./data/pinecone_subset", output_path="output/pinecone_reconstruction", num_zones=3, radial_clip=0.7)

## 4. Visualization

In [ ]:
# Visualization Logic here

## 5. Citations\n```bibtex\n@article{sun2026vggt3,\n  title={VGG-T³: Offline Feed-Forward 3D Reconstruction at Scale},\n  author={Sun, Aljoša and others},\n  journal={arXiv preprint arXiv:2602.23361},\n  year={2026}\n}\n```